In [2]:
from pathlib import Path
import random
import csv


# =========================================================
# CONFIGURATION
# =========================================================

IMAGE_DIR = Path(
    r"C:\Users\vivek chauhan\Desktop\dsa babbar\Videos\archive (26)\last_dataset\images"
)

MASK_DIR = Path(
    r"C:\Users\vivek chauhan\Desktop\dsa babbar\Videos\archive (26)\last_dataset\masks"
)

OUTPUT_DIR = Path(r"C:\liverCTSegmentation\outputs\splits")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEED = 42


In [3]:


# =========================================================
# GET ALL IMAGES
# =========================================================

image_paths = sorted(
    IMAGE_DIR.glob("*.png")
)

print("=" * 60)
print("CREATING VOLUME-LEVEL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)

print("Total images:", len(image_paths))


# =========================================================
# GROUP IMAGES BY VOLUME
# =========================================================

volume_to_images = {}

for image_path in image_paths:

    parts = image_path.stem.split("-")

    if len(parts) < 3:
        continue

    volume_id = f"{parts[0]}-{parts[1]}"

    if volume_id not in volume_to_images:
        volume_to_images[volume_id] = []

    volume_to_images[volume_id].append(image_path)



CREATING VOLUME-LEVEL TRAIN / VALIDATION / TEST SPLIT
Total images: 58638


In [4]:

# =========================================================
# GET VOLUME IDs
# =========================================================

volume_ids = sorted(
    volume_to_images.keys()
)

print("Total volumes:", len(volume_ids))


# =========================================================
# SHUFFLE VOLUMES
# =========================================================

random.seed(SEED)

random.shuffle(volume_ids)


# =========================================================
# 70 / 15 / 15 SPLIT
# =========================================================

total_volumes = len(volume_ids)

train_count = int(total_volumes * 0.70)

val_count = int(total_volumes * 0.15)

test_count = total_volumes - train_count - val_count


train_volumes = volume_ids[:train_count]

val_volumes = volume_ids[
    train_count:train_count + val_count
]

test_volumes = volume_ids[
    train_count + val_count:
]


Total volumes: 131


In [5]:


# =========================================================
# COLLECT IMAGE PATHS
# =========================================================

train_images = []

val_images = []

test_images = []


for volume_id in train_volumes:
    train_images.extend(
        volume_to_images[volume_id]
    )


for volume_id in val_volumes:
    val_images.extend(
        volume_to_images[volume_id]
    )


for volume_id in test_volumes:
    test_images.extend(
        volume_to_images[volume_id]
    )


# =========================================================
# SORT IMAGE PATHS
# =========================================================

train_images = sorted(train_images)

val_images = sorted(val_images)

test_images = sorted(test_images)


In [6]:


# =========================================================
# PRINT SPLIT INFORMATION
# =========================================================

print("\n" + "=" * 60)
print("VOLUME SPLIT")
print("=" * 60)

print(
    "Training volumes:",
    len(train_volumes)
)

print(
    "Validation volumes:",
    len(val_volumes)
)

print(
    "Testing volumes:",
    len(test_volumes)
)


print("\n" + "=" * 60)
print("IMAGE SPLIT")
print("=" * 60)

print(
    "Training images:",
    len(train_images)
)

print(
    "Validation images:",
    len(val_images)
)

print(
    "Testing images:",
    len(test_images)
)

print(
    "Total images:",
    len(train_images) +
    len(val_images) +
    len(test_images)
)




VOLUME SPLIT
Training volumes: 91
Validation volumes: 19
Testing volumes: 21

IMAGE SPLIT
Training images: 39677
Validation images: 8960
Testing images: 10001
Total images: 58638


In [7]:

# =========================================================
# PERCENTAGES
# =========================================================

total_images = len(image_paths)

print("\n" + "=" * 60)
print("IMAGE PERCENTAGES")
print("=" * 60)

print(
    "Training:",
    round(len(train_images) / total_images * 100, 2),
    "%"
)

print(
    "Validation:",
    round(len(val_images) / total_images * 100, 2),
    "%"
)

print(
    "Testing:",
    round(len(test_images) / total_images * 100, 2),
    "%"
)



IMAGE PERCENTAGES
Training: 67.66 %
Validation: 15.28 %
Testing: 17.06 %


In [8]:


# =========================================================
# CHECK FOR VOLUME LEAKAGE
# =========================================================

train_set = set(train_volumes)

val_set = set(val_volumes)

test_set = set(test_volumes)


print("\n" + "=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

print(
    "Train ∩ Validation:",
    len(train_set.intersection(val_set))
)

print(
    "Train ∩ Test:",
    len(train_set.intersection(test_set))
)

print(
    "Validation ∩ Test:",
    len(val_set.intersection(test_set))
)




LEAKAGE CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [9]:

# =========================================================
# SAVE SPLIT CSV FILES
# =========================================================

def save_split(
    filename,
    volume_list,
    image_list
):

    output_file = OUTPUT_DIR / filename

    with open(
        output_file,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        writer.writerow(
            [
                "volume_id",
                "image_path",
                "mask_path"
            ]
        )

        for image_path in image_list:

            parts = image_path.stem.split("-")

            volume_id = (
                f"{parts[0]}-{parts[1]}"
            )

            mask_path = MASK_DIR / image_path.name

            writer.writerow(
                [
                    volume_id,
                    str(image_path),
                    str(mask_path)
                ]
            )

    print(
        "Saved:",
        output_file
    )


save_split(
    "train.csv",
    train_volumes,
    train_images
)

save_split(
    "validation.csv",
    val_volumes,
    val_images
)

save_split(
    "test.csv",
    test_volumes,
    test_images
)



Saved: C:\liverCTSegmentation\outputs\splits\train.csv
Saved: C:\liverCTSegmentation\outputs\splits\validation.csv
Saved: C:\liverCTSegmentation\outputs\splits\test.csv


In [10]:

# =========================================================
# SAVE VOLUME LISTS
# =========================================================

def save_volume_list(
    filename,
    volumes
):

    output_file = OUTPUT_DIR / filename

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        for volume in volumes:
            f.write(volume + "\n")

    print(
        "Saved:",
        output_file
    )


save_volume_list(
    "train_volumes.txt",
    train_volumes
)

save_volume_list(
    "validation_volumes.txt",
    val_volumes
)

save_volume_list(
    "test_volumes.txt",
    test_volumes
)



Saved: C:\liverCTSegmentation\outputs\splits\train_volumes.txt
Saved: C:\liverCTSegmentation\outputs\splits\validation_volumes.txt
Saved: C:\liverCTSegmentation\outputs\splits\test_volumes.txt


In [11]:

# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n" + "=" * 60)
print("SPLIT COMPLETED")
print("=" * 60)

print("Random seed:", SEED)

print(
    "Train volumes:",
    len(train_volumes)
)

print(
    "Validation volumes:",
    len(val_volumes)
)

print(
    "Test volumes:",
    len(test_volumes)
)

print(
    "\nSplit files saved to:",
    OUTPUT_DIR
)


SPLIT COMPLETED
Random seed: 42
Train volumes: 91
Validation volumes: 19
Test volumes: 21

Split files saved to: C:\liverCTSegmentation\outputs\splits
